# IQM QEC Pipeline Showcase

This notebook demonstrates the main pipeline results we have been building:

1. Clean distance-3 rotated surface code on the best IQM Emerald placement using `iqm_qec_pipeline`.
2. Distance-3 Snakes-and-Ladders deformed surface codes with a deliberately faulty coupler using `snl_iqm_pipeline`.
3. Synthetic distance-5 Snakes-and-Ladders code with a central data-qubit defect, using calibration-derived noise.

Each experiment produces:

- a logical error probability sweep over rounds,
- a fitted logical error rate per round,
- a linear-scale and log-scale plot,
- a lattice/stabilizer visualization.

Set `RUN_MODE = "hardware"` only when you want to submit hardware jobs. The synthetic mode uses the calibration-derived Stim noise model and is the safest way to sanity-check the pipeline quickly.

In [67]:
import importlib
import os
import subprocess
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

SNL_REPO_URL = "https://github.com/amazon-science/snakes_and_ladders_adapting_the_surface_code_to_defects.git"
SNL_REPO_DIR = Path("snakes_and_ladders_adapting_the_surface_code_to_defects")
if not (SNL_REPO_DIR / "defects_module").exists():
    print("Downloading Snakes-and-Ladders dependency...")
    subprocess.run(["git", "clone", "--depth", "1", SNL_REPO_URL, str(SNL_REPO_DIR)], check=True)
os.environ.setdefault("SNL_REPO", str(SNL_REPO_DIR.resolve()))
if str(SNL_REPO_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(SNL_REPO_DIR.resolve()))

import iqm_qec_pipeline as iqm
import snl_iqm_pipeline as snl
import helper_visualisation as hv

iqm = importlib.reload(iqm)
snl = importlib.reload(snl)
hv = importlib.reload(hv)

# Global run controls. Use hardware only when you are ready to submit real jobs.
RUN_MODE = "synthetic"      # "synthetic" or "hardware"
API_URL = "https://resonance.iqm.tech/"
TOKEN = os.environ.get("IQM_TOKEN")  # or set TOKEN = "..." in a private cell
SHOTS = 5_000
ROUND_VALUES = (1, 2, 3, 5, 8, 10)
BASIS = "Z"
DECODER_NOISE = "average"     # "exact" or "average"

CAL = iqm.refresh_calibration_if_needed(
    api_url=API_URL,
    token=TOKEN,
    quantum_computer="emerald",
)
print("Using calibration:", CAL)

python(77463) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Cloning into 'snakes_and_ladders_adapting_the_surface_code_to_defects'...


No IQM_TOKEN set; using existing calibration data: calibration_data/2026-06-05T06_19_42.975934Z.json
Using calibration: calibration_data/2026-06-05T06_19_42.975934Z.json


In [ ]:
TOKEN = ""    # set your IQM token here

## 1. Distance 3 Surface Code on IQM Emerald

This uses the `iqm_qec_pipeline`. It optimizes the d=3 layout on the IQM Emerald lattice, runs the round sweep, plots the fitted logical error rate, and visualizes the stabilizer-measurement couplings.

In [ ]:
clean_d3 = iqm.run_round_sweep(
    distance=3,
    round_values=ROUND_VALUES,
    shots=SHOTS,
    run_mode="hardware",
    decoder_noise="average",
    cal=CAL,
    token=TOKEN,
    api_url=API_URL,
    optimize_layout=True,
    refresh_calibration=False,
    show_error_plot=True,
)

clean_d3["summary"]


In [ ]:
clean_d3_patch_fig = hv.plot_used_emerald_patch(
    clean_d3["results"][0],
    title="Clean d=3 optimized IQM Emerald patch",
)



In [ ]:
clean_d3_timeslice_fig = hv.plot_iqm_timeslice_layers(
    clean_d3["results"][0],
    max_layers=8,
    cols=4,
    title="Clean d=3 single-round gate/reset/measurement timeslices on IQM Emerald",
    zoom_to_used=True,
)
clean_d3_timeslice_fig


## 2. Distance 3 deformed surface code along defective coupler

This section intentionally runs on the IQM hardware. It uses a fixed real Emerald patch around the top-center faulty region in the online device view: the faulty `QB45_QB46` coupler while keeping QB46 usable. The helper below returns the exact `sw_offset` and defect arguments, and then the sweep runs with `optimize_layout=False` so it does not move away from this physical region.


In [ ]:
faulty_region = snl.hardware_faulty_region_demo_kwargs(
    distance=3,
    defect_qubits=[],
    defect_couplers=["QB45_QB46"],
    cal=CAL,
)
print(faulty_region["summary"])

snl_d3_faulty_coupler = snl.run_snl_round_sweep(
    distance=faulty_region["distance"],
    round_values=ROUND_VALUES,
    shots=SHOTS,
    mode="hardware",
    defect_qubits=faulty_region["defect_qubits"],
    defect_couplers=faulty_region["defect_couplers"],
    use_calibration_defects=faulty_region["use_calibration_defects"],
    basis=BASIS,
    cal=CAL,
    token=TOKEN,
    api_url=API_URL,
    sw_offset=faulty_region["sw_offset"],
    optimize_layout=False,
    show_lattice_plot=True,
    show_error_plot=True,
)

snl_d3_faulty_coupler["summary"]


In [ ]:
snl_d3_faulty_coupler["error_figure"]

In [ ]:
snl_d3_region_result = snl_d3_faulty_coupler["results"][0]
print("used IQM qubits:", sorted(hv.used_iqm_qubits(snl_d3_region_result)))
print("used IQM couplers:", sorted(hv.used_iqm_couplers(snl_d3_region_result)))
print("defective couplers:", snl_d3_region_result["summary"]["defect_couplers"])

snl_d3_hardware_region_fig = hv.plot_snl_emerald_hardware_region(
    snl_d3_region_result,
    title="d=3 SnL placement using the QB45-QB46 defective coupler region",
    show_full_lattice=True,
)

## 3. Synthetic d=5 SnL Code With a Central Data-Qubit Defect

This is a simulation using the calibration-derived noise model. We mark central `QB28` as unavailable to force a Snakes-and-Ladders deformation. This is intended as the larger-lattice synthetic sanity check, not a hardware submission.

In [ ]:
snl_d5_center_defect = snl.run_snl_round_sweep(
    distance=5,
    round_values=ROUND_VALUES,
    shots=SHOTS,
    mode="synthetic",
    defect_qubits=[27],
    use_calibration_defects=False,
    basis=BASIS,
    cal=CAL,
    optimize_layout=True,
    show_lattice_plot=True,
    show_error_plot=True,
)

snl_d5_center_defect["summary"]

## Compare the Main Fitted Logical Error Rates

This compact table compares the per-round logical error rates extracted from the three examples.

In [ ]:
main_results = {
    "clean d=3 IQM pipeline": clean_d3["summary"],
    "SnL d=3 faulty coupler": snl_d3_faulty_coupler["summary"],
    "SnL d=5 center defect synthetic": snl_d5_center_defect["summary"],
}

for name, summary in main_results.items():
    print(name)
    print("  corrected eps_L:  ", summary.get("logical_error_rate"))
    print("  uncorrected eps_L:", summary.get("uncorrected_logical_error_rate"))
    print("  effective distance:", summary.get("effective_distance"))
    print("  defects:", summary.get("defect_qubits"), summary.get("defect_couplers"))

## Tutorial Use Cases

This cell gives small synthetic examples of how to call the two pipelines. Each pipeline has one example for a single logical error probability measurement and one example for extracting a logical error rate from a round sweep.

In [ ]:
# Tutorial settings: small synthetic jobs so this cell can be run without submitting hardware jobs.
TUTORIAL_DISTANCE = 3          # surface-code distance d
TUTORIAL_ROUNDS = 3            # number of QEC rounds for one logical-error-probability estimate
TUTORIAL_ROUND_VALUES = (1, 2, 3)  # round counts used to fit logical error rate per round
TUTORIAL_SHOTS = 500           # shots per experiment; increase for smoother estimates
TUTORIAL_BASIS = "Z"           # memory basis: "Z" or "X"
TUTORIAL_DECODER_NOISE = "average"  # decoder noise model: "average" or "exact"


# -----------------------------------------------------------------------------
# 1. Base IQM pipeline: one logical error probability measurement
# -----------------------------------------------------------------------------
# iqm.run_pipeline parameters:
#   distance: code distance.
#   rounds: number of surface-code cycles.
#   shots: number of sampled/hardware shots.
#   basis: logical memory basis, "Z" or "X".
#   cal: calibration JSON used for layout/noise.
#   run_mode: "synthetic" samples from calibration noise; "hardware" submits to IQM.
#   decoder_noise: "average" compact calibration model or "exact" qubit/coupler-specific model.
#   refresh_calibration: if True, try to fetch today's IQM calibration data.
#   save_stim_file: if True, write the generated Stim circuit into stim_files/.
#   optimize_layout: choose best IQM Emerald placement; useful/automatic for hardware.
base_probability = iqm.run_pipeline(
    distance=TUTORIAL_DISTANCE,
    rounds=TUTORIAL_ROUNDS,
    shots=TUTORIAL_SHOTS,
    basis=TUTORIAL_BASIS,
    cal=CAL,
    run_mode="synthetic",
    decoder_noise=TUTORIAL_DECODER_NOISE,
    refresh_calibration=False,
    save_stim_file=False,
    optimize_layout=False,
)
print("Base pipeline logical error probability:", base_probability["summary"]["corrected_ler"])


# -----------------------------------------------------------------------------
# 2. Base IQM pipeline: logical error rate extraction from a round sweep
# -----------------------------------------------------------------------------
# iqm.run_round_sweep parameters are the same as run_pipeline, except:
#   round_values: list/tuple of rounds to run separately.
#   show_error_plot: if True, return the fitted linear/log logical-error plot.
# The fitted per-round logical error rate is summary["logical_error_rate"].
base_sweep = iqm.run_round_sweep(
    distance=TUTORIAL_DISTANCE,
    round_values=TUTORIAL_ROUND_VALUES,
    shots=TUTORIAL_SHOTS,
    basis=TUTORIAL_BASIS,
    cal=CAL,
    run_mode="synthetic",
    decoder_noise=TUTORIAL_DECODER_NOISE,
    refresh_calibration=False,
    save_stim_file=False,
    optimize_layout=False,
    show_error_plot=True,
)
print("Base pipeline logical error rate per round:", base_sweep["summary"]["logical_error_rate"])


# -----------------------------------------------------------------------------
# 3. Snakes-and-Ladders pipeline: one logical error probability measurement
# -----------------------------------------------------------------------------
# snl.run_snl_pipeline parameters:
#   distance, rounds, shots, basis, cal: same meaning as the base pipeline.
#   mode: "synthetic" or "hardware".
#   num_qubit_defects / num_coupler_defects: randomly place this many unavailable components.
#   defect_qubits: explicit IQM qubits to treat as unavailable, e.g. [27] or ["QB27"].
#   defect_couplers: explicit IQM couplers to treat as unavailable, e.g. ["QB45_QB46"].
#   use_calibration_defects: include genuinely unavailable components from calibration/backend.
#   optimize_layout: scan Emerald placements and choose the best valid SnL patch.
#   sw_offset: fixed patch placement when optimize_layout=False.
#   show_plot: return a lattice visualization.
snl_probability = snl.run_snl_pipeline(
    distance=TUTORIAL_DISTANCE,
    rounds=TUTORIAL_ROUNDS,
    shots=TUTORIAL_SHOTS,
    mode="synthetic",
    defect_couplers=["QB45_QB46"],
    use_calibration_defects=False,
    basis=TUTORIAL_BASIS,
    cal=CAL,
    sw_offset=(3, 1),
    optimize_layout=False,
    refresh_calibration=False,
    show_plot=True,
)
print("SnL logical error probability:", snl_probability["summary"]["logical_error_probability"])


# -----------------------------------------------------------------------------
# 4. Snakes-and-Ladders pipeline: logical error rate extraction from a round sweep
# -----------------------------------------------------------------------------
# snl.run_snl_round_sweep parameters match run_snl_pipeline, except:
#   round_values: list/tuple of rounds to run separately on the same deformed patch.
#   show_lattice_plot: return the SnL lattice/defect visualization.
#   show_error_plot: return the fitted linear/log logical-error plot.
# The fitted per-round logical error rate is summary["logical_error_rate"].
snl_sweep = snl.run_snl_round_sweep(
    distance=TUTORIAL_DISTANCE,
    round_values=TUTORIAL_ROUND_VALUES,
    shots=TUTORIAL_SHOTS,
    mode="synthetic",
    defect_couplers=["QB45_QB46"],
    use_calibration_defects=False,
    basis=TUTORIAL_BASIS,
    cal=CAL,
    sw_offset=(3, 1),
    optimize_layout=False,
    refresh_calibration=False,
    show_lattice_plot=True,
    show_error_plot=True,
)
print("SnL logical error rate per round:", snl_sweep["summary"]["logical_error_rate"])
